# Task 5.1: Feature Engineering — Create New Features

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

df = pd.read_excel('Practice_Dataset.xlsx')

# 1. Interaction features - combine two columns
# Example: if you have hours and rate, create total_pay
# df['total_pay'] = df['hours'] * df['rate']

# 2. Binning - convert numeric to categories
# Example: group punch_count into Low/Medium/High
df['punch_level'] = pd.cut(df['punch_count'],
                            bins=[0, 2, 4, float('inf')],
                            labels=['Low', 'Medium', 'High'])
print(df['punch_level'].value_counts())

# 3. Date features (if you have date columns)
df['work_date'] = pd.to_datetime(df['work_date'])
df['day_of_week'] = df['work_date'].dt.dayofweek
df['month'] = df['work_date'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# 4. Flag features - binary indicators
df['is_absent'] = (df['status'] == 'ABSENT').astype(int)
print(df[['status', 'is_absent']].head())

punch_level
Medium    195
Low        67
High       63
Name: count, dtype: int64
    status  is_absent
0  PRESENT          0
1  PRESENT          0
2  PRESENT          0
3  PRESENT          0
4  PRESENT          0


# Task 5.2: Build an sklearn Pipeline

In [2]:
# A Pipeline chains preprocessing steps so you can apply them
# consistently to training AND test data (no data leakage!)

# Separate features by type
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include='object').columns.tolist()

# Drop high-cardinality ID/free-text columns from the categorical pipeline
# (badge_number, name = unique identifiers; in_time/out_time = raw time strings)
categorical_features = [c for c in categorical_features
                         if c not in ['badge_number', 'name', 'in_time', 'out_time']]

# Numeric pipeline: fill missing -> scale
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Categorical pipeline: fill missing -> one-hot encode
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# Combine both into a ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

# Fit and transform
X_clean = preprocessor.fit_transform(df)
print(f'Original shape: {df.shape}')
print(f'Cleaned shape: {X_clean.shape}')

# Get feature names after encoding
cat_names = preprocessor.named_transformers_['cat']\
    .named_steps['encoder'].get_feature_names_out(categorical_features)
all_features = list(numeric_features) + list(cat_names)
print(f'Feature names: {all_features}')

Original shape: (360, 13)
Cleaned shape: (360, 15)
Feature names: ['punch_count', 'day_of_week', 'month', 'is_weekend', 'is_absent', 'position_ACCOUNTANT', 'position_ADMINISTRATOR', 'position_CLERK', 'position_DRIVER', 'position_ENGINEER', 'position_LABORER', 'position_SUPERVISOR', 'position_TECHNICIAN', 'status_ABSENT', 'status_PRESENT']


# Task 5.3: Export the Clean Dataset

In [5]:
# Convert the cleaned array back to a DataFrame
df_clean = pd.DataFrame(X_clean, columns=all_features)
print('Clean dataset summary:')
print(f'  Shape: {df_clean.shape}')
print(f'  Missing values: {df_clean.isnull().sum().sum()}')
print(f'  Dtypes: {df_clean.dtypes.value_counts().to_dict()}')

# Save the clean dataset
df_clean.to_csv('clean_dataset.csv', index=False)
print('Saved clean_dataset.csv')

# Also save the preprocessor for reuse in Week 4
import joblib
joblib.dump(preprocessor, 'preprocessor.pkl')
print('Saved preprocessor.pkl')

Clean dataset summary:
  Shape: (360, 15)
  Missing values: 0
  Dtypes: {dtype('float64'): 15}
Saved clean_dataset.csv
Saved preprocessor.pkl


# Task 5.4: Cleaning Report

DATA CLEANING REPORT - Practice Dataset

Date: 16-07-2026

Analyst: Fajer Alshammari

ORIGINAL DATASET:
- Rows: 360, Columns: 8
- Missing values: 70 (in_time, out_time only)
- Outliers detected: 0 (IQR and Z-score both agree, punch_count only numeric col)

CLEANING STEPS APPLIED:
1. Missing Values: Filled in_time/out_time with mode (most frequent time)
   because they are categorical time strings, not true numeric data.
2. Outliers: Checked with IQR + Z-score on punch_count - none found,
   so no capping was actually needed (pipeline still applies it safely).
3. Encoding: One-Hot on position (nominal, 8 categories), Label on status
   (binary PRESENT/ABSENT).
4. Scaling: StandardScaler on numeric features (punch_count, day_of_week,
   month, is_weekend, is_absent).
5. Feature Engineering: punch_level (binned), day_of_week, month,
   is_weekend (from work_date), is_absent (flag from status).

FINAL DATASET:
- Rows: 360, Columns: 15
- Missing values: 0
- All features numeric and scaled

FILES PRODUCED:
- clean_dataset.csv   - model-ready dataset
- preprocessor.pkl    - reusable sklearn pipeline
- cleaning_log.csv    - decisions and rationale